In [55]:
import pandas as pd

# ---- Load dataset ----
file_path = "historical_data/revision/1000_sample_historical_data_train_profile_nn_sp_adult.csv"
df = pd.read_csv(file_path)

# ---- Basic sanity check ----
print("Columns:", df.columns.tolist())

# Make sure column exists
assert 'utility_sp' in df.columns, "utility_sp column not found!"

# ---- Min and Max values ----
min_value = df['utility_sp'].min()
max_value = df['utility_sp'].max()
print("\nMinimum utility_sp:", min_value)
print("Maximum utility_sp:", max_value)

# ---- Rows corresponding to min and max ----
min_row = df.loc[df['utility_sp'].idxmin()]
max_row = df.loc[df['utility_sp'].idxmax()]

print("\nRow with MIN utility_sp:\n")
print(min_row)

print("\nRow with MAX utility_sp:\n")
print(max_row)


Columns: ['sampling', 'invalid_value', 'missing_value', 'floating_point', 'unit_converter', 'distribution_shape', 'multicollinearity', 'normalization', 'outlier', 'punctuation', 'stopword', 'lowercase', 'whitespace', 'model', 'outlier_bef_normalization_strat', 'outlier_bef_outlier_strat', 'diff_sensitive_attr', 'ratio_sensitive_attr', 'cov', 'class_imbalance_ratio', 'corr_Age', 'ot_Age', 'corr_Workclass', 'ot_Workclass', 'corr_fnlwgt', 'ot_fnlwgt', 'corr_Education', 'ot_Education', 'corr_Education_Num', 'ot_Education_Num', 'corr_Martial_Status', 'ot_Martial_Status', 'corr_Occupation', 'ot_Occupation', 'corr_Relationship', 'ot_Relationship', 'corr_Race', 'ot_Race', 'corr_Sex', 'ot_Sex', 'corr_Capital_Gain', 'ot_Capital_Gain', 'corr_Capital_Loss', 'ot_Capital_Loss', 'corr_Hours_per_week', 'ot_Hours_per_week', 'corr_Country', 'ot_Country', 'utility_sp']

Minimum utility_sp: -0.0142877590455061
Maximum utility_sp: 0.2796124318742259

Row with MIN utility_sp:

sampling                      

In [57]:
# Count rows where utility_sp == 0
zero_count = (df['utility_sp'] == 0).sum()

print("Number of rows where utility_sp = 0:", zero_count)

Number of rows where utility_sp = 0: 0


In [62]:
import pandas as pd

# Path to your dataset
file_path = "historical_data/revision/1000_sample_historical_data_train_profile_nn_sp_adult.csv"

# Load
df = pd.read_csv(file_path)

print("Rows before cleaning:", len(df))

# Remove rows where utility_sp = 0
df = df[df['utility_sp'] > 0]

print("Rows after cleaning:", len(df))

# Overwrite the same file
df.to_csv(file_path, index=False)

print("Dataset updated successfully.")


Rows before cleaning: 1000
Rows after cleaning: 725
Dataset updated successfully.


In [ ]:
import pandas as pd

train_path = "historical_data/revision/1000_sample_historical_data_train_profile_nn_sp_adult.csv"
test_path  = "historical_data/revision/1000_sample_historical_data_test_profile_nn_sp_adult.csv"

key_cols = [
    "sampling","invalid_value","missing_value","floating_point","unit_converter",
    "distribution_shape","multicollinearity","normalization","outlier","punctuation",
    "stopword","lowercase","whitespace","model"
]

# Load datasets
train = pd.read_csv(train_path)
test  = pd.read_csv(test_path)

# Ensure numeric
train["utility_sp"] = pd.to_numeric(train["utility_sp"], errors="coerce")
test["utility_sp"]  = pd.to_numeric(test["utility_sp"], errors="coerce")

# ---- Create lookup dictionary from TRAIN ----
# Key = tuple(pipeline configuration), Value = train utility
train_lookup = {}

for _, row in train.iterrows():
    key = tuple(row[col] for col in key_cols)
    train_lookup[key] = row["utility_sp"]

# ---- Filter TEST rows ----
selected_rows = []

for _, row in test.iterrows():

    key = tuple(row[col] for col in key_cols)

    # skip if configuration not found in train
    if key not in train_lookup:
        continue

    train_utility = train_lookup[key]
    test_utility  = row["utility_sp"]

    # apply condition
    if (test_utility > 0.05) and (train_utility < 0.05 and train_utility > 0):
        selected_rows.append(row)

# ---- Create new dataset from TEST rows only ----
filtered_test = pd.DataFrame(selected_rows)

print("Filtered rows from test dataset:", len(filtered_test))

# ---- Save separate dataset ----
filtered_test.to_csv("historical_data/revision/filtered/0.1_filtered_adult_nn_sp.csv", index=False)

#print("Saved: filtered_opposite_generalization_cases.csv")


Filtered rows from test dataset: 64
